This notebook cleans the data obtained from SerpAPI i.e., the raw JSON files.
To check that this code works simply upload the raw Guardian and Daily Mail JSONs (found in data directory) and then run all the cells in order.
This will leave you with final_data.json which can be used in the analysis notebook.

(Steps 2 and 3 should be done manually:
  To remove the empty article just search for body: "" and remove it
  To remove the irrelevant articles, search for their titles (which are stored in data/non-uber-articles.json))

In [3]:
# Pip installs
!pip3 install beautifulsoup4
!pip3 install requests

# imports
from bs4 import BeautifulSoup
import requests
#import serpapi
import json
from datetime import datetime
import dateutil.parser as parser
import pandas as pd
import json

In [1]:

# This block of code will convert the extant JSON files into a the final one for analysis (as shown in Template.JSON)

# DAILY MAIL ARTICLE -> FORMAT
# Sanity checks
# Only one headline
# Only one (group) of authors
# Only one date

# First time accessing an article

def read_dailymail_article(url):
  err_obj = {
      'url': url
  }
  # all daily mail articles contain article in url so ignore non-articles
  if 'article' not in url:
    err_obj['failed'] = 'non-article url'
    return err_obj

  # Requests need the url of the page to access
  site = requests.get(url)

  # Beautiful Soup takes the text from the page accessed by requests and makes it easy to use
  # 'html.parser' is the standard way to process HTML in python
  soup = BeautifulSoup(site.text, 'html.parser')

  # Get headline - CHECK THAT THERE IS ONLY ONE HEADLINE


  headline = soup.find_all('h1')
  if len(headline) != 1:
    err_obj['failed'] = 'headline'
    return err_obj

  headline = headline[0].get_text().lower()

  # sometimes headline starts with exclusive without a space, so add a space if so
  if headline.startswith('exclusive'):
    headline = 'exclusive ' + headline.split('exclusive', maxsplit=2)[1].strip()

 # there can be multiple authors
  authors = soup.find_all(class_="author")
  # sometimes authors stored in this tag author-section byline-plain so try this too
  by_for = False
  if len(authors) == 0:
    by_for = True
    authors = soup.find_all(class_="author-section byline-plain")

  # sometimes this tag is used to store multiple authors separated by AND author-section mol-para-with-font byline-plain
  and_separator = False
  if len(authors) == 0:
    and_separator = True
    by_for = False
    authors = soup.find_all(class_="author-section mol-para-with-font byline-plain")

  # check there's at least one
  if len(authors) == 0:
    err_obj['failed'] = 'author'
    return err_obj

  authors = [a.get_text().lower().strip() for a in authors]
  # and_separator medians multiple authors separated by and
  if(by_for):
    authors_tmp = authors
    authors = []
    for author in authors_tmp:
      [authors.append(a.strip()) for a in author.split("by ", maxsplit=2)[1].split(' for ', maxsplit=2)[0].split(' and ')]

  if(and_separator):
    authors_tmp = authors
    authors = []
    for author in authors_tmp:
      [authors.append(a.strip()) for a in author.split("by ", maxsplit=2)[1].split(' and ')]

  # Dailymail has published and updated date let's record both
  # Get published time
  published = soup.find(class_ = 'article-timestamp article-timestamp-published')
  if published is None:
    err_obj['failed'] = 'published'
    return err_obj
  published = published.find('time').attrs['datetime']
  published = parser.parse(published)
  # Convert to datetime
  # Get updated time
  updated = soup.find(class_ = 'article-timestamp article-timestamp-updated')
  if updated is not None:
    updated = updated.find('time').attrs['datetime']
    updated = parser.parse(updated)
  else:
    updated = published

  # Get body
  # Ignore subheadings - inconsistent and not always super related to article e.g., it might check out this podcast
  # Ignore captions for photos (will just be repeats from article)
  paragraphs = soup.find_all('p', class_ = 'mol-para-with-font')

  body = ' '.join([p.get_text().strip() for p in paragraphs])

  # Sometimes dailymail articles e.g. https://www.dailymail.co.uk/news/article-2546104/Uber-exposed-dirty-tactics-admitting-attempt-poach-drivers-rival-likely-aggressive.html
  # 's bodies don't have the class, they're just in p tags
  # below code fixes this
  if body.strip() == "":
    paragraphs = soup.find_all('p')
    body = ' '.join([p.get_text().strip() for p in paragraphs])


  # put everything in lower case for purposes of checking for dupes

  return {
      "headline": headline.lower(),
      "authors": authors,
      "published": published.timestamp(),
      "updated": updated.timestamp(),
      "body": body.lower(),
      "url": url
  }

def read_guardian_article(url):
  # Requests need the url of the page to access
  site = requests.get(url)

  # Can't validate URL for guardian because it doesn't contain special string for articles

  # Beautiful Soup takes the text from the page accessed by requests and makes it easy to use
  # 'html.parser' is the standard way to process HTML in python
  soup = BeautifulSoup(site.text, 'html.parser')

  err_obj = {
      'url': url
  }

  # Get headline - CHECK THAT THERE IS ONLY ONE HEADLINE
  headline = soup.find_all('h1')
  if len(headline) != 1:
    err_obj['failed'] = 'headline'
    return err_obj
  headline = headline[0].get_text().lower()


  # Validate authors
  authors = soup.find_all('a', {'rel': 'author'})

  if len(authors) != 0: authors = [author.get_text().strip().lower() for author in authors]

  # Sometimes authors stored elsewhere
  if len(authors) == 0:
    author = soup.find('meta', {'property': 'article:author'})
    if author is not None:
      authors = [author.attrs['content'].strip().lower()]


  if len(authors) == 0:
    err_obj['failed'] = 'author'
    return err_obj



  # Dailymail has published and updated date let's record both
  # Get published time
  published = soup.find('meta', {'property': 'article:published_time'})

  if published is None:
    err_obj['failed'] = 'published'
    return err_obj
  published = published.attrs['content']
  published = parser.parse(published)

  # Get updated time
  updated = soup.find('meta', {'property': 'article:modified_time'})

  if updated is not None:
    updated = updated.attrs['content']
    updated = parser.parse(updated)
  else:
    updated = published

  # Get body
  # Include subheadings - check if it exists first though

  subheading = soup.find_all(class_ = "dcr-1m3qdf6")
  if len(subheading) > 0:
    subheading = subheading[0].get_text().strip() + " "
  else:
    subheading = ""

  # Ignore captions for photos (will just be repeats from article)
  paragraphs = (soup.find_all('p', class_ = "dcr-s3ycb2"))

  body = subheading + ' '.join([p.get_text().strip() for p in paragraphs])

  # put everything in lower case for purposes of checking for dupes

  return {
      "headline": headline.lower(),
      "authors": authors,
      "published": published.timestamp(),
      "updated": updated.timestamp(),
      "body": body.lower(),
      "url": url
  }


In [4]:
# Using the read daily mail and read guardian functions - let's put our current list into a nicer format
# IF ERROR: update the location paths inside the convert_all_results function

def convert_all_results(paper):
  out = []
  loc = ""

  if paper == 'dailymail':
    loc = './raw_dailymail.json'
  elif paper == 'guardian':
    loc = './raw_guardian.json'
  else:
    return []

  with open(loc, 'r') as f:
    data = json.load(f)
    i = 0
    failed = 0
    for page in data:
      for article in page['organic_results']:
        i += 1
        if paper == 'dailymail': obj = read_dailymail_article(article['link'])
        elif paper == 'guardian': obj = read_guardian_article(article['link'])
        if 'failed' in obj:
          print(obj['failed'] + " : " + article['link'])
          failed += 1
        print(str(i) + " completed, " + str(failed) + " failed")


        out.append(obj)
  return out

# store the cleaned SerpAPI results in new files
# this converts search results -> list of articles of each newspaper
for paper in ['dailymail', 'guardian']:
  with open("clean_" + paper + '.json', 'w') as f:
    json.dump(convert_all_results(paper), f)

1 completed, 0 failed
non-article url : https://www.dailymail.co.uk/sciencetech/uber/index.html
2 completed, 1 failed
3 completed, 1 failed
non-article url : https://www.dailymail.co.uk/sciencetech/uber/index.html?page=19
4 completed, 2 failed
5 completed, 2 failed
6 completed, 2 failed
7 completed, 2 failed
8 completed, 2 failed
non-article url : https://www.dailymail.co.uk/sciencetech/uber/index.html?page=3
9 completed, 3 failed
10 completed, 3 failed
11 completed, 3 failed
12 completed, 3 failed
13 completed, 3 failed
14 completed, 3 failed
15 completed, 3 failed
16 completed, 3 failed
17 completed, 3 failed
18 completed, 3 failed
19 completed, 3 failed
20 completed, 3 failed
21 completed, 3 failed
22 completed, 3 failed
non-article url : https://www.dailymail.co.uk/video/tvshowbiz/video-3083039/Video-Michael-Cera-struggles-grill-burgers-Uber-Eats-commercial.html
23 completed, 4 failed
24 completed, 4 failed
25 completed, 4 failed
26 completed, 4 failed
27 completed, 4 failed
non-ar

In [5]:
# This function merges the clean daily mail and guardian lists
def merge_clean_data(dailymail_loc, guardian_loc):
  merged = {
      'time': datetime.now().timestamp()
  }
  # Read dailymail and put into the merged object
  with open(dailymail_loc, 'r', encoding='utf-8') as f:
    merged['dailymail'] = json.load(f)
    # Remove failed non-articles
    merged['dailymail'] = list(filter(lambda a: ('failed' not in a), merged['dailymail']))
  # Read guardian and put into the merged object
  with open(guardian_loc, 'r', encoding='utf-8') as f:
    merged['guardian'] = json.load(f)
    # Remove failed non-articles
    merged['guardian'] = list(filter(lambda a: ('failed' not in a), merged['guardian']))

  return merged

# Store merged into a file
with open('./merged.json', 'w') as f:
  json.dump(merge_clean_data('./clean_dailymail.json', './clean_guardian.json'), f)

In [6]:
# Remove duplicates in the dataset
loc = "./merged.json"

# check for dupe headlines

obj = []
with open('./merged.json', encoding='utf-8') as f:
    obj = json.load(f)

headlines = {
    'guardian': set([]),
    'dailymail': set([])
}

# Loop through articles
n = 0
new_obj = {'guardian' : [], 'dailymail' : []}

for paper in ['guardian', 'dailymail']:
    for article in obj[paper]:
        headline = article['headline'].strip()
        if(headline in headlines[paper]):
            print("DUPE: " + headline)
            n += 1
        else:
            headlines[paper].add(headline)
            new_obj[paper].append(article)

print(n)

with open('./merged_nodupe1.json', 'w') as f:
  json.dump(new_obj, f)

DUPE: uber facing new uk driver claims of racial discrimination
DUPE: more than 10,000 london black-cab drivers launch £250m uber lawsuit
DUPE: uber users in uk will be able to book flights on app by summer
DUPE: union calls on uk uber users to join 24-hour strike over revelations
DUPE: us couple blocked from suing uber after crash say daughter agreed to terms while using uber eats
DUPE: osborne, hancock and other ministers did not declare secret uber meetings
DUPE: the uber files: firm knew it launched illegally in australia, then leaned on governments to change the law
DUPE: uber ‘tech bros’ sought to destroy australian taxi app using corporate espionage, court hears
DUPE: uber must overhaul london business model after high court ruling
DUPE: uber drivers strike over pay and conditions
DUPE: uber settles vat claim with hmrc and posts better than expected results
DUPE: uber aims for greener trips and to expand london electric vehicle fleet
DUPE: uber drivers strike over pay and condit

In [7]:
# gonna check for dupes by making dictionary of all authors (and list of authors), and making sure no two similar articles show

obj = []
with open('./merged_nodupe1.json', encoding='utf-8') as f:
    obj = json.load(f)

authors_dict = {}

for paper in ['guardian', 'dailymail']:
    for article in obj[paper]:
        index = str([a.strip() for a in article['authors']])
        headline = article['headline'].strip()
        if index not in authors_dict: authors_dict[index] = [headline]
        else: authors_dict[index].append(headline)

# we're only interested in duplicates so remove all authors with multiple entries


n = 0
new_obj = {}

for key in authors_dict.keys():
    if len(authors_dict[key]) < 2:
        n += 1
    else:
        new_obj[key] = authors_dict[key]

print("removed " + str(n) + " authors")

# store remaining authors in file for inspection


with open('./dupe_authors.json', 'w') as f:
  json.dump(new_obj, f)



removed 223 authors


In [8]:
# remove 2025 articles
with open('./merged_nodupe1.json', encoding='utf-8') as f:
    obj = json.load(f)
    new_obj = {'guardian' : [], 'dailymail' : []}
    for paper in ['guardian', 'dailymail']:
        for article in obj[paper]:
          if article['published'] < 1735689600:
            new_obj[paper].append(article)
          else:
            print(article['headline'])
    with open('./final_data.json', 'w') as f:
      json.dump(new_obj, f)

taxi firms crowdfund legal battle with uber over vat on fares in uk
uk app drivers: will you be logging off on valentine’s day?
moment uber driver is filmed with a phone in his hand during the middle of a ride as he calls his daughter... to wish her happy birthday while behind the wheel
uber driver horrified after mother left her child alone in the car for 8 minutes to ensure he didn't drive off at her stop
thousands of illegal immigrants banned from uber eats after company starts crackdown
exclusive uber eats customer calls out disgraceful act after making shock discovery
shameful karen caught falsely accusing male uber driver of sexual harassment for refusing to drive faster
i ordered from uber eats and received a note with a very bizarre request from the restaurant
exclusive furious customer claims uber driver used viral photo of vomit to scam him into paying £80 fine - have you also fallen victim to this trick?
uber driver sparks outrage after series of errors caused passenger to m